# Ancient Text Generation

If a word is surprising, the model is bad.

## Problem definition

Before transformers, before word embeddings, a language model predicted the next word by counting how often it followed the previous n-1 words.

This is an n-gram language model. A raw count-based model assigns zero probability to anything it has not seen, Fifty years of smoothing research fixed that.

### Basic Concept

#### N-gram probability

```
P(w_i | w_{i-n+1}, ..., w_{i-i}) =
    count(content, w) / count(content)
```

n typeically 3 for trigrams, 4 for 4-grams

#### Zero-count problem.

Smoothing approaches.
1. Laplace. Add 1 to every count.
2. Good-Turning. Reallocate probability mass from higher-frequency events to unseen ones.
3. Interpolation. Combine n-gram, (n - 1)-gram, etc.
4. Backoff. If n-gram has count zero, fallback to (n-1)-gram
5. Absolute discounting. Subtract a fixed discount D from all counts, redistribute to unseen.
6. Kneser-Ney. Absolute discounting plus a clever choice for the lower-order model: use continuation probability (how many contexts a word appears in) instead of raw frequency.

### Evaluation: Perplexity.

The exponent of the average negative log-likelihood per word on a held-out test set.

```
perplexity = exp(-(1/N) * sum(log(P(w_i | context_i))))
```

A perplexity of 100 means that the model is as confused as it would be choosing uniformly among 100 words.

# Build your Own

## Trigram counts

In [1]:
from collections import Counter, defaultdict

def train_ngram(corpus_tokens, n = 3):
    ngrams = Counter()
    contexts = Counter()
    for sentence in corpus_tokens:
        padded = ["<s>"] * (n - 1) + sentence + ["</s>"]
        for i in range(len(padded) - n + 1):
            ctx = tuple(padded[i:i+n-1])
            word = padded[i + n - 1]
            ngrams[ctx + (word,)] += 1
            contexts[ctx] += 1
    return ngrams, contexts

def raw_probability(ngrams, contexts, context, word):
    ctx = tuple(context)
    if contexts.get(ctx, 0) == 0:
        return 0.0
    return ngrams.get(ctx + (word,), 0) / contexts[ctx]



## Laplace smoothing

In [2]:
def laplace_probability(ngrams, contexts, vocab_size, context, word):
    ctx = tuple(context)
    numerator = ngrams.get(ctx + (word,), 0) + 1
    denominator = contexts.get(ctx, 0) + vocab_size
    return numerator / denominator

## Kneser-Ney

In [ ]:
def kneser_ney_bigram_model(corpus_tokens, discount=0.75):
    """
    训练 Kneser-Ney 二元语言模型。

  核心思想（相比普通 bigram）：
  1. 绝对折扣（Absolute Discounting）：每个见过的 bigram 计数减 D，减出来的概率质量分给“没见过”的词。
  2. 续接概率（Continuation Probability）：回退时不用 unigram 原始频率 P(w)，
     而用“w 出现在多少种不同前文后面”——更能反映词的多功能性。
     例："Francisco" 虽只跟在 "San" 后，但 "San" 能接很多词，所以 P(San|Francisco) 应更高。
    """
    unigrams = Counter()          # 单词出现总次数（本实现里主要辅助统计）
    bigrams = Counter()           # (前文, 当前词) -> 共现次数
    unigram_contexts = defaultdict(set)  # 每个词 w -> 出现在它前面的不同词集合

    for sentence in corpus_tokens:
        padded = ["<s>"] + sentence + ["</s>"]  # 句首 <s> 提供起始上下文
        for i, w in enumerate(padded):
            unigrams[w] += 1
            if i > 0:
                prev = padded[i - 1]
                bigrams[(prev, w)] += 1
                # 记录：w 曾被哪些不同的 prev 词接在后面（续接上下文数）
                unigram_contexts[w].add(prev)

    # 续接概率 P_cont(w) = |{前文 : bigram(前文, w) 存在}| / 所有不同 bigram 类型总数
    # 分母是所有词的“前文种类数”之和，即语料中不同 bigram 类型的总数
    total_unique_bigrams = sum(len(ctx_set) for ctx_set in unigram_contexts.values())
    continuation_prob = {
        w: len(ctx_set) / total_unique_bigrams for w, ctx_set in unigram_contexts.items()
    }

    # 每个前文 prev 后面总共跟了多少个词（含重复），用于归一化
    context_totals = Counter()
    for (prev, w), count in bigrams.items():
        context_totals[prev] += count

    # 每个前文 prev 后面跟过多少种不同的词（去重），用于计算折扣系数 λ
    unique_follow = defaultdict(set)
    for (prev, w) in bigrams:
        unique_follow[prev].add(w)

    def prob(prev, w):
        """P_KN(w | prev) = 折扣后的直接估计 + λ(prev) * P_cont(w)"""
        count = bigrams.get((prev, w), 0)       # c(prev, w)
        denom = context_totals.get(prev, 0)     # c(prev, ·) 总和
        if denom == 0:
            # 前文从未见过，完全依赖续接概率
            return continuation_prob.get(w, 1e-9)

        # 第一项：绝对折扣 ML 估计  max(c - D, 0) / c(prev)
        first_term = max(count - discount, 0) / denom

        # λ(prev) = D * |{w : c(prev,w)>0}| / c(prev)
        # 把减掉的概率质量，按续接概率分给所有可能的后续词
        lambda_prev = discount * len(unique_follow[prev]) / denom

        return first_term + lambda_prev * continuation_prob.get(w, 1e-9)

    return prob

## Generating text with sampling

In [ ]:
import random

def generate(prob_fn, vocab, prefix, max_len=30, seed=0):
    rng = random.Random(seed)
    tokens = list(prefix)
    for _ in range(max_len):
        candidates = [(w, prob_fn(tokens[-1], w)) for w in vocab]
        total = sum(p for _, p in candidates)
        r = rng.random() * total
        acc = 0.0
        for w, p in candidates:
            acc += p
            if r <= acc:
                tokens.append(w)
                break
        if tokens[-1] == "</s>":
            break
    
    return tokens

## Perplexity

In [ ]:
import math

def perplexity(prob_fn, sentences):
    total_log_prob = 0.0
    total_tokens = 0
    for sentence in sentences:
        padded = ["<s>"] + sentence + ["</s>"]
        for i in range(1, len(padded)):
            p = prob_fn(padded[i - 1], padded[i])
            total_log_prob += math.log(max(p, 1e-12))
            total_tokens += 1
    return math.exp(-total_log_prob / total_tokens)